In [ ]:
!pip install -q \
langchain==0.3.27 \
langchain-community==0.3.29 \
langchain-text-splitters==0.3.9 \
langchain-huggingface==0.1.2 \
sentence-transformers==3.2.1 \
faiss-cpu==1.11.0 \
transformers==4.46.3 \
torch==2.5.1 \
accelerate==1.1.1 \
pymupdf==1.25.1 \
pypdf==5.1.0 \
python-docx==1.1.2 \
pandas==2.2.3 \
numpy==2.0.2 \
streamlit==1.41.1 \
tqdm==4.67.1

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

In [ ]:
!pip install -q langchain-google-genai==2.0.11

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 32.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.6 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0.6.18 which is incompatible.


In [ ]:
import os
import fitz
import faiss
import numpy as np
import pandas as pd

from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_core.prompts import ChatPromptTemplate
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("LangChain:", langchain.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)

Torch: 2.5.1+cu124
Transformers: 4.46.3
LangChain: 0.3.27
Sentence Transformers: 3.2.1
FAISS: 1.11.0
Pandas: 2.2.3
NumPy: 2.0.2


In [ ]:
import torch
import torchvision
import transformers

print(torch.__version__)
print(torchvision.__version__)
print(transformers.__version__)

2.5.1+cu124
0.20.1+cu124
4.46.3


In [ ]:
import kagglehub
path = kagglehub.dataset_download("rohanthoma/ebook-pdfs")

Using Colab cache for faster access to the 'ebook-pdfs' dataset.


In [ ]:
import os

print(os.listdir(path))

['College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'ConceptsofBiology-WEB-19-602.pdf', 'Introduction_to_Political_Science_-_WEB-19-549.pdf']


In [ ]:
for root, dirs, files in os.walk(path):
    print(root)
    for file in files:
        print("   ", file)

/kaggle/input/ebook-pdfs
    College_Physics_2e-WEB_7Zesafu-23-1473.pdf
    ConceptsofBiology-WEB-19-602.pdf
    Introduction_to_Political_Science_-_WEB-19-549.pdf


In [ ]:
print(os.listdir(path))

['College_Physics_2e-WEB_7Zesafu-23-1473.pdf', 'ConceptsofBiology-WEB-19-602.pdf', 'Introduction_to_Political_Science_-_WEB-19-549.pdf']


In [ ]:
pdf_files = []

for file in os.listdir(path):
    if file.endswith(".pdf"):
        pdf_files.append(os.path.join(path, file))

print("Number of PDFs :", len(pdf_files))

for pdf in pdf_files:
    print(pdf)

Number of PDFs : 3
/kaggle/input/ebook-pdfs/College_Physics_2e-WEB_7Zesafu-23-1473.pdf
/kaggle/input/ebook-pdfs/ConceptsofBiology-WEB-19-602.pdf
/kaggle/input/ebook-pdfs/Introduction_to_Political_Science_-_WEB-19-549.pdf


In [ ]:
documents = []

for pdf in pdf_files:

    doc = fitz.open(pdf)

    text = ""

    for page in doc:
        text += page.get_text()

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": os.path.basename(pdf)
            }
        )
    )

print(len(documents))

3


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=500,

    chunk_overlap=100,

    separators=["\n\n","\n"," ",""]

)

chunks = text_splitter.split_documents(documents)

print("Total Chunks :", len(chunks))

Total Chunks : 16296


In [ ]:
for i in range(3):

    print("="*80)

    print(chunks[i].metadata)

    print(chunks[i].page_content)

{'source': 'College_Physics_2e-WEB_7Zesafu-23-1473.pdf'}
INTRODUCTION TO SCIENCE AND THE REALM OF PHYSICS, PHYSICAL QUANTITIES, AND UNITS
CHAPTER 1
Introduction: The Nature of Science and
Physics
1.1 Physics: An Introduction
1.2 Physical Quantities and Units
1.3 Accuracy, Precision, and Significant Figures
1.4 Approximation
What is your
first reaction when you hear the word “physics”? Did you imagine working through difficult equations or memorizing
{'source': 'College_Physics_2e-WEB_7Zesafu-23-1473.pdf'}
formulas that seem to have no real use in life outside the physics classroom? Many people come to the subject of
physics with a bit of fear. But as you begin your exploration of this broad-ranging subject, you may soon come to
realize that physics plays a much larger role in your life than you first thought, no matter your life goals or career
choice.
Consider the Veil Nebula, a cloud of heated dust and gas located about 2,400 light years from Earth (a light year is
{'source': 'Colleg

In [ ]:
embedding_model = HuggingFaceEmbeddings(

    model_name="BAAI/bge-small-en-v1.5",

    model_kwargs={
        "device":"cpu"
    },

    encode_kwargs={
        "normalize_embeddings":True
    }

)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_db = FAISS.from_documents(

    documents=chunks,

    embedding=embedding_model

)

In [28]:
vector_db.save_local("vector_database")

In [29]:
vector_db = FAISS.load_local(

    "vector_database",

    embedding_model,

    allow_dangerous_deserialization=True

)



In [54]:
def retrieve(question, k=3):

    results = vector_db.similarity_search_with_score(
        question,
        k=k
    )

    return results

In [55]:
for i, result in enumerate(results, start=1):

    print("=" * 100)
    print(f"Result {i}")
    print("Source :", result.metadata["source"])
    print()
    print(result.page_content[:1000])

Result 1
Source : College_Physics_2e-WEB_7Zesafu-23-1473.pdf

Newton’s second law of motion states that the
acceleration of a system is directly proportional to
and in the same direction as the net external force
acting on the system, and inversely proportional to
its mass.
•
In equation form, Newton’s second law of motion
is
.
•
This is often written in the more familiar form:
.
•
The weight
of an object is defined as the force of
gravity acting on an object of mass
. The object
experiences an acceleration due to gravity
:
•
Result 2
Source : College_Physics_2e-WEB_7Zesafu-23-1473.pdf

Newton’s second law. (See Figure 4.20(c).) Newton’s third law may be used to identify whether forces are exerted
between components of a system (internal) or between the system and something outside (external). As illustrated
earlier in this chapter, the system of interest depends on what question we need to answer. This choice becomes
easier with practice, eventually developing into an almost unconscio

In [46]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [48]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [56]:
evaluation_prompt = """
You are a Retrieval Evaluator.

Question:
{question}

Retrieved Context:
{context}

Evaluate whether this context is sufficient to answer the question.

Return exactly in this format:

Relevant: Yes or No
Score: number between 0 and 100
"""

In [57]:
import re

def evaluate(question, retrieved_results):

    context = "\n\n".join(
        [doc.page_content for doc, _ in retrieved_results]
    )

    prompt = evaluation_prompt.format(
        question=question,
        context=context
    )

    response = llm.invoke(prompt)

    text = response.content

    try:
        score = int(
            re.search(r"Score:\s*(\d+)", text).group(1)
        )
    except:
        score = 0

    return score

In [51]:
from google import genai

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [52]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=userdata.get("GOOGLE_API_KEY"),
    temperature=0
)

In [58]:
response = llm.invoke("Say hello in one sentence.")

print(response.content)

Hello, how can I help you today?


In [59]:
rewrite_prompt = """
Rewrite the question to improve document retrieval.

Return ONLY the rewritten question.

Question:

{question}
"""

In [60]:
def rewrite(question):

    response = llm.invoke(

        rewrite_prompt.format(

            question=question

        )

    )

    return response.content.strip()

In [61]:
answer_prompt = """
You are an AI Assistant.

Answer ONLY using the provided context.

If the answer does not exist, reply exactly:

I couldn't find the answer in the uploaded documents.

Context:

{context}

Question:

{question}

Answer:
"""

In [62]:
def answer(question, retrieved_results):

    context = "\n\n".join(
        [doc.page_content for doc, _ in retrieved_results]
    )

    prompt = answer_prompt.format(

        context=context,

        question=question

    )

    response = llm.invoke(prompt)

    return response.content

In [68]:

def corrective_rag(question):

    print("=" * 100)
    print("Original Question:")
    print(question)

    retrieved = retrieve(question)

    similarity_scores = [score for _, score in retrieved]

    evaluator_score = evaluate(question, retrieved)

    print(f"\nRetrieval Evaluation Score: {evaluator_score}/100")


    if evaluator_score < 70:

        print("\nWeak Retrieval Detected")
        print("Rewriting Query...")

        rewritten_question = rewrite(question)

        print("\nRewritten Question:")
        print(rewritten_question)

        question = rewritten_question

        retrieved = retrieve(question)

        similarity_scores = [score for _, score in retrieved]

        evaluator_score = evaluate(question, retrieved)

        print(f"\nNew Retrieval Score: {evaluator_score}/100")

    else:

        print("\nRetrieval Quality: Good")

    print(f"\nRetrieved Chunks: {len(retrieved)}")


    similarity = np.mean(similarity_scores)

    confidence = round(
        (evaluator_score * 0.6)
        +
        ((1 - similarity) * 100 * 0.4),
        2
    )


    final_answer = answer(question, retrieved)

    docs = [doc for doc, _ in retrieved]

    return final_answer, docs, confidence

In [72]:
question = "What is Newton's Second Law?"

answer_text, docs, confidence = corrective_rag(question)
print("="*100)

print("Confidence:", confidence,"%")
print("=" * 100)
print("Sources:\n")

sources = sorted({doc.metadata["source"] for doc in docs})

for source in sources:
    print(f"- {source}")

Original Question:
What is Newton's Second Law?

Retrieval Evaluation Score: 95/100

Retrieval Quality: Good

Retrieved Chunks: 3
Confidence: 81.98 %
Sources:

- College_Physics_2e-WEB_7Zesafu-23-1473.pdf


In [73]:
import shutil

shutil.make_archive(
    "vector_database",
    "zip",
    "vector_database"
)

'/content/vector_database.zip'